  Cell 1 — Markdown (concept)
  # Null / None in Pandas DataFrames                                                                                                                                                      
                                    
  - `None` — Python's null object (object dtype)                                                                                                                                          
  - `np.nan` — IEEE float "Not a Number" (float dtype)                                                                                                                                    
  - `pd.NaT` — "Not a Time", for datetime nulls                                                                                                                                           
  - Pandas treats both None and np.nan as missing in most operations     

In [12]:
import pandas as pd
import numpy as np
data = {
      "name":   ["Alice", "Bob", None,    "David"],
      "age":    [25,      np.nan, 30,      22],    
      "score":  [88.5,    92.0,  np.nan,  np.nan],                                                                                                                                        
      "joined": pd.to_datetime(["2021-01-01", "2021-06-15", None, "2022-03-10"])     
}

df = pd.DataFrame(data)                                                                                                                                                                 
print(df)                                                                                                                                                                               
print(df.dtypes)

    name   age  score     joined
0  Alice  25.0   88.5 2021-01-01
1    Bob   NaN   92.0 2021-06-15
2   None  30.0    NaN        NaT
3  David  22.0    NaN 2022-03-10
name              object
age              float64
score            float64
joined    datetime64[ns]
dtype: object


In [13]:
  # Check individual cells                                                                                                                                                                
df.isnull()        # True where null
df.notnull()       # True where NOT null                                                                                                                                                
                                                                                                                                                                                        
# Count nulls per column
df.isnull().sum() 

name      1
age       1
score     2
joined    1
dtype: int64

In [ ]:
# Check individual cells                                                                                                                                                                
df.isnull()        # True where null
df.notnull()       # True where NOT null                                                                                                                                                
                                                                                                                                                                                        
# # Count nulls per column
df.isnull().sum(axis=0)       
#   The chain works like this:
                                                                                                                                                                                          
#   df.isnull()    →  DataFrame of True/False  (same shape as df)
#          .sum()  →  Series  (True counts as 1, False as 0, summed per column)                                                                                                             

                                                                                                                                                                                          
#   If you want a single number (total nulls in the whole df):                                                                                                                              
                                                                                                                                                                                          
#   df.isnull().sum().sum()   # Series → scalar                                                                                                                                             
                                                                                                                                                                                          
#   Or to get nulls per row instead of per column:                                                                                                                                          
                                                                                                                                                                                          
#   df.isnull().sum(axis=1)   # still a Series, but indexed by row number                                                                                                                   
                                                                                                                                                                                                                                                                 
                
# # Null percentage per column                                                                                                                                                            
df.isnull().mean() * 100    
                        
# # Any null in entire df?                                                                                                                                                                
df.isnull().values.any()

,name,age,score,joined
1,Bob,NaN,92.0,2021-06-15
2,None,30.0,NaN,NaT
3,David,22.0,NaN,2022-03-10


In [ ]:
                                                                                                                                                                                        
df.isnull().sum(axis=1)                                                                                                                                                                 
# 0    0        
# 1    1                                                                                                                                                                                
# 2    2
# 3    1                                                                                                                                                                                
# dtype: int64  

df.isnull().values.any(axis=1)
# array([False,  True,  True,  True])
                                                                                                                                                                                        
# To actually show rows with nulls, the second one is what you use as a filter:                                                                                                           
                                                                                                                                                                                        
df[df.isnull().values.any(axis=1)]      # rows that have at least 1 null                                                                                                                
                                                                                                                                                                                        
# You can also do it without .values:                                                                                                                                                     
                                                                                                                                                                                        
df[df.isnull().any(axis=1)]             # same result, more pandas-native                                                                                                               
                                                                                                                                                                                        
# And the count version is useful for sorting by "most broken" rows:                                                                                                                      
                                                                                                                                                                                        
df[df.isnull().sum(axis=1) > 0]         # rows with at least 1 null                                                                                                                     
df.loc[df.isnull().sum(axis=1) >= 2]    # rows with 2 or more nulls  

In [ ]:
#   Cell 4 — Drop nulls
df.dropna()                        # drop rows with ANY null                                                                                                                            
df.dropna(how="all")               # drop rows where ALL are null
df.dropna(subset=["age"])          # drop only if 'age' is null                                                                                                                         
df.dropna(thresh=3)                # keep rows with at least 3 non-nulls

In [ ]:
#   Cell 5 — Fill nulls     
import pandas as pd                                                                                                                                                                     
import numpy as np
                                                                                                                                                                                        
df = pd.DataFrame({                                                                                                                                                                     
    "name":  ["Alice", "Bob", None, "David", None],
    "age":   [25, np.nan, 30, np.nan, 22],                                                                                                                                              
    "score": [np.nan, 92.0, np.nan, 88.0, 75.0]                                                                                                                                         
})                                                                                                                                                                                      
print(df)                                                                                                                                                                               
#     name   age  score                                                                                                                                                                 
# 0  Alice  25.0    NaN
# 1    Bob   NaN   92.0                                                                                                                                                                 
# 2   None  30.0    NaN
# 3  David   NaN   88.0                                                                                                                                                                 
# 4   None  22.0   75.0                                                                                                                                                                 

# 1. fill with mean                                                                                                                                                                     
df["age"].fillna(df["age"].mean())                                                                                                                                                      
# 0    25.0
# 1    25.666...   ← was NaN, filled with mean of [25, 30, 22]                                                                                                                          
# 2    30.0                                                                                                                                                                             
# 3    25.666...   ← was NaN
# 4    22.0                                                                                                                                                                             
                
# 2. fill with median                                                                                                                                                                   
df["score"].fillna(df["score"].median())
# 0    88.0    ← was NaN, median of [92, 88, 75] = 88.0
# 1    92.0                                                                                                                                                                             
# 2    88.0    ← was NaN
# 3    88.0                                                                                                                                                                             
# 4    75.0     
                                                                                                                                                                                        
# 3. fill with constant
df["name"].fillna("Unknown")                                                                                                                                                            
# 0      Alice  
# 1        Bob
# 2    Unknown   ← was None
# 3      David                                                                                                                                                                          
# 4    Unknown   ← was None
                                                                                                                                                                                        
# 4. forward fill — copies the last known value downward
df["age"].ffill()                                                                                                                                                                       
# 0    25.0
# 1    25.0    ← copied from row 0                                                                                                                                                      
# 2    30.0                                                                                                                                                                             
# 3    30.0    ← copied from row 2
# 4    22.0                                                                                                                                                                             
                
# 5. backward fill — copies the next known value upward
df["age"].bfill()                                                                                                                                                                       
# 0    25.0
# 1    30.0    ← copied from row 2                                                                                                                                                      
# 2    30.0                                                                                                                                                                             
# 3    22.0    ← copied from row 4
# 4    22.0                                                                                                                               
                                                        

In [ ]:
# Cell 6 — None vs NaN gotchas                                                                                                                                                            
# None equality check doesn't work like you expect                                                                                                                                      
None == None       # True in Python               
np.nan == np.nan   # FALSE! NaN is never equal to itself    
type(None)     # <class 'NoneType'>    
#  s = None                                                                                                                                                                                
#   - None is a real object sitting in memory
#   - s is a reference that points to that object
#   - Python creates None once at startup and reuses it forever (singleton)                                                                                                                 
   
#   s  →  [None object at 0x7f...] in memory                                                                                                                                                
                                                                                                                                                                                          
#   id(None)   # 9789744 — it has a real memory address                                                                                                                                  
                                                        
# Correct way to check                                                                                                                                                                  
pd.isnull(None)    # True
pd.isnull(np.nan)  # True                                                                                                                                                               
pd.isna(np.nan)    # True  (same as isnull)
                                                                                                                                                                                        
x = np.nan                                                                                                                                                                              
x is np.nan        # True (identity check works)                                                                                                                                        
                                                                                                                                                                                          
  

In [ ]:
                                                                                                                                                                                   
#   Cell 7 — Practical: replace and check
df_clean = df.copy()                                                                                                                                                                    
df_clean["age"].fillna(df_clean["age"].median(), inplace=True)
df_clean["score"].fillna(0, inplace=True)                                                                                                                                               
df_clean["name"].fillna("Unknown", inplace=True)
                                                                                                                                                                                        
print("Nulls remaining:", df_clean.isnull().sum().sum())
                                                                                                                                                                                          
#   ---             
#   Key things to remember:                                                                                                                                                                 
#   - Always use pd.isnull() / pd.isna() — never == None or == np.nan
#   - dropna() vs fillna() — choose based on how much data you can afford to lose
#   - np.nan is a float, so a column with np.nan in an int column gets cast to float64      

In [24]:
print(0.1 + 0.2) #0.30000000000000004
print(round(0.1 + 0.2, 2)) #0.30

0.30000000000000004
0.3


In [ ]:
                                                                                                                                                                                        
class Dog:
    def __init__(self, name):                                                                                                                                                           
        self.name = name

    def __eq__(self, other):                                                                                                                                                            
        return self.name == other.name
    
d1 = Dog("Rex")
d2 = Dog("Rex")
d1 == d2
hash(d1)        # TypeError: unhashable type: 'Dog'  ✗                                                                                                                                  
{d1}            # TypeError ✗
{d1: "val"}     # TypeError ✗   


True

In [ ]:
# Setup:
import pandas as pd                                                                                                                                                                     
                
df = pd.DataFrame({
"name":  ["Alice", "Bob", "Charlie"],
"age":   [25, 30, 35],               
"score": [88.5, 92.0, 78.0]                                                                                                                                                         
}, index=["a", "b", "c"])      
                                                                                                                                                                                        
#        name  age  score
# a     Alice   25   88.5                                                                                                                                                               
# b       Bob   30   92.0
# c   Charlie   35   78.0                                                                                                                                                               
                
# 
# df.loc["a", "name"]       # 'Alice'                                                                                                                                                     
# df.loc["a"]               # entire row a — returns Series
# df.loc[:, "age"]          # entire age column                                                                                                                                           
# df.loc["a":"b", "name":"age"]   # slice rows and columns by label                                                                                                                       
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# 2. iloc — integer position based (like array index)                                                                                                                                     
# df.iloc[0, 0]             # 'Alice'   — row 0, col 0                                                                                                                                    
# df.iloc[0]                # entire first row — returns Series
# df.iloc[:, 1]             # entire second column                                                                                                                                        
# df.iloc[0:2, 0:2]         # first 2 rows, first 2 cols
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# 3. at — single cell by label (faster than loc for one cell)
# df.at["a", "name"]        # 'Alice'                                                                                                                                                     
# df.at["b", "score"]       # 92.0   
                                                                                                                                                                                        
# ---                                                                                                                                                                                     
# 4. iat — single cell by position (faster than iloc for one cell)
# df.iat[0, 0]              # 'Alice'                                                                                                                                                     
# df.iat[1, 2]              # 92.0
                                                                                                                                                                                        
# ---             

SyntaxError: invalid character '—' (U+2014) (3132572186.py, line 22)

In [ ]:
import pandas as pd                                                                                                                                                                     
                
df = pd.DataFrame({
"name":  ["Alice", "Bob", "Charlie"],
"age":   [25, 30, 35],               
"score": [88.5, 92.0, 78.0]                                                                                                                                                         
}) 
# df.iloc[0, 1:3]
# df.loc[0, ["name","score"]]
#   df[] only accepts:                                                                                                                                                                      
#   df["name"]          # column name — works ✓                                                                                                                                             
#   df[0:2]             # row slice — works ✓                                                                                                                                               
#   df[0]               # KeyError ✗ — not a column name                                                                                                                                    
#   df[0, "name"]       # KeyError ✗ — not valid syntax for df[]    

#   ---                                                                                                                                                                                     
#   To mix position and label, you need loc or iloc:                                                                                                                                        
                                                                                                                                                                                          
#   # you want row 0, column "name"                                                                                                                                                       
                                                                                                                                                                                          
#   df.loc[0, "name"]       # ✓ — row by index label 0, col by name                                                                                                                         
#   df.iloc[0, 0]           # ✓ — row by position 0, col by position 0   

name     Alice
score     88.5
Name: 0, dtype: object

In [45]:
df = pd.DataFrame({
"name":  ["Alice", "Bob", "Alice", "Charlie"],
"dept":  ["HR", "IT", "HR", "IT"],            
"score": [88, 92, 88, 75]
})

df["dept"].unique()         # array(['HR', 'IT'])  — returns numpy array                                                                                                                
df["name"].unique()         # array(['Alice', 'Bob', 'Charlie'])      
   
# nunique() — count of unique values                                                                                                                                                      
df["dept"].nunique()        # 2  — scalar                                                                                                                                               
df.nunique()                # count per column — returns Series
# name     3                                                                                                                                                                            
# dept     2                                                                                                                                                                            
# score    3     
# 
# value_counts() — unique values with their frequency
df["dept"].value_counts()                                                                                                                                                               
# HR    2                
# IT    2                                                                                                                                                                               
                
df["name"].value_counts()                                                                                                                                                               
# Alice      2
# Bob        1                                                                                                                                                                          
# Charlie    1  


# drop_duplicates() — remove duplicate rows
df.drop_duplicates()                    # remove fully duplicate rows                                                                                                                   
df.drop_duplicates(subset=["dept"])     # keep first row per unique dept
df.drop_duplicates(subset=["dept"], keep="last")   # keep last instead             

,name,dept,score
2,Alice,HR,88
3,Charlie,IT,75
